# 🕵️ Aula 18 — Detecção e Diagnóstico de Anomalias

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** `reator_4falhas.csv` — 2 anos de dados de reator com 4 falhas

---

## Contexto

Modelos supervisionados precisam de dados rotulados. **E se nunca houve uma falha para rotular?** Métodos **não-supervisionados** aprendem o que é "normal" e detectam o que é "diferente" — sem saber qual é cada falha.

## Três Detectores

| Método | Assinatura do "normal" | Score de anomalia |
|--------|------------------------|-------------------|
| **PCA** | Direções de maior variância | Q-residual (distância ao modelo) e T² |
| **Isolation Forest** | Profundidade de isolamento | s(x) = 2^(−h/c) |
| **Autoencoder** | Reconstrução dos dados normais | Erro de reconstrução (MSE) |

## 3.1 — Treinar 3 Detectores no Reator

Treine **só nos 60% iniciais** (período normal) e avalie nos **40% finais** (que contêm as 4 falhas).

### Passo 1: Carregar + preparar dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import f1_score, confusion_matrix

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula18/reator_4falhas.csv"
df = pd.read_csv(URL)
features = ['T_reator_C','P_reator_bar','vazao_refri_L_min','conversao','U_trocador_W_m2K']
X = df[features].values
y = (df['rotulo_falha'] > 0).astype(int)   # rótulos só p/ avaliar no fim

split = int(0.6 * len(X))
sc = StandardScaler().fit(X[:split])
X_tr = sc.transform(X[:split])
X_te = sc.transform(X[split:])
y_te = y[split:].values
print(f"Treino (normal): {len(X_tr)} | Teste (com falhas): {len(X_te)}")

### Passo 2: PCA (Q-residual)

In [ ]:
pca = PCA(n_components=3).fit(X_tr)
Xr_tr = pca.inverse_transform(pca.transform(X_tr))
Q_tr = np.sum((X_tr - Xr_tr)**2, axis=1)
th_Q = np.percentile(Q_tr, 99)              # threshold do TREINO (normal)

Xr_te = pca.inverse_transform(pca.transform(X_te))
Q_te = np.sum((X_te - Xr_te)**2, axis=1)
pred_PCA = (Q_te > th_Q).astype(int)
print(f"PCA: F1={f1_score(y_te, pred_PCA):.2f}")

### Passo 3: Isolation Forest

In [ ]:
iforest = IsolationForest(random_state=42, contamination=0.01)
iforest.fit(X_tr)
sc_tr = -iforest.score_samples(X_tr)
sc_te = -iforest.score_samples(X_te)
th_IF = np.percentile(sc_tr, 99)
pred_IF = (sc_te > th_IF).astype(int)
print(f"Isolation Forest: F1={f1_score(y_te, pred_IF):.2f}")

### Passo 4: Autoencoder (MLP)

In [ ]:
# Autoencoder: comprime para latente e reconstrói
# Shape: 5 -> 10 -> 5 -> 3 -> 5 -> 10 -> 5
ae = MLPRegressor(hidden_layer_sizes=(10, 5, 3, 5, 10), max_iter=500, random_state=42)
ae.fit(X_tr, X_tr)   # treinado só nos NORMais → reconstrói mal anomalias

e_tr = np.sum((X_tr - ae.predict(X_tr))**2, axis=1)
th_AE = np.percentile(e_tr, 99)
e_te = np.sum((X_te - ae.predict(X_te))**2, axis=1)
pred_AE = (e_te > th_AE).astype(int)
print(f"Autoencoder: F1={f1_score(y_te, pred_AE):.2f}")

### Passo 5: Comparar matrizes de confusão por falha

In [ ]:
tl = df['rotulo_falha'][split:].values
print(f"{'Detector':<16}", *[f"F{i}: %det" for i in range(1,5)], " F1")
for nome, pred in [('PCA', pred_PCA), ('IsolationForest', pred_IF), ('Autoencoder', pred_AE)]:
    det = "  ".join(f"{pred[tl==f].mean()*100:5.0f}%" for f in [1,2,3,4])
    print(f"{nome:<18}{det}  {f1_score(y_te, pred)*100:.1f}%")

print("\nMatriz de confusão (Autoencoder):")
print(confusion_matrix(y_te, pred_AE))

### Passo 6: Visualizar scores ao longo do tempo

In [ ]:
plt.figure(figsize=(12, 4))
plt.semilogy(np.arange(len(e_te)), e_te, label='Autoencoder (erro reconstr.)')
plt.axhline(th_AE, color='red', ls='--', label='Threshold')
# sombrear falhas
for f, color in [(1,'red'),(2,'orange'),(3,'purple'),(4,'pink')]:
    idx = np.where(tl==f)[0]
    if len(idx): plt.axvspan(idx[0], idx[-1], color=color, alpha=0.15)
plt.xlabel('Ponto no teste'); plt.ylabel('Erro de reconstrução')
plt.legend(); plt.grid(alpha=0.3); plt.title('Erro de reconstrução do Autoencoder')
plt.tight_layout(); plt.show()

### ✏️ Pausa reflexiva (2 min)

Por que métodos diferentes detectam falhas diferentes? (PCA = padrão linear global; IF = isolamento; AE = reconstrução)

## 3.2 — Exercício em Grupo: Threshold e Custo

Cada grupo testa um percentil de threshold diferente.

| Grupo | Percentil | Detector |
|-------|-----------|----------|
| **A** | 90 | PCA |
| **B** | 95 | Isolation Forest |
| **C** | 99 | Autoencoder |
| **D** | 99.5 | Autoencoder |

**Custo:** falso negativo (falha não detectada) = 10× o falso positivo (alarme falso).

In [ ]:
percentil = 99   # ← mude para o do seu grupo
# detector = seu grupo

# Exemplo: Autoencoder com percentil escolhido
th_g = np.percentile(e_tr, percentil)
pred_g = (e_te > th_g).astype(int)

tn, fp, fn, tp = confusion_matrix(y_te, pred_g).ravel()
custo = fn*10 + fp*1   # falso negativo custa 10x
print(f"Percentil {percentil}: F1={f1_score(y_te, pred_g):.2f}  custo={custo}")
print(f"  TP={tp}  FP={fp} (alarmes falsos)  FN={fn} (falhas perdidas)  TN={tn}")

> **Perguntas:**
> 1. Qual percentil minimiza o custo total?
> 2. Threshold alto = especificidade alta, mas falhas passam. Vale a pena?
> 3. Como calibrar sem rótulos? (simulação com falhas injetadas)

### 🧠 Desafio extra (NT)

1% de alarmes falsos em 10.000 sensores = 100 alarmes/dia → operador ignora. Mas threshold alto demais → falha real passa. Como calibrar sem dados rotulados?

> _Escreva aqui..._

---

## Checklist

- [ ] Treino SÓ com dados normais (60%)
- [ ] PCA (Q-residual/T²) treinado
- [ ] Isolation Forest treinado
- [ ] Autoencoder treinado
- [ ] Threshold documentado (percentil)
- [ ] Matriz de confusão por falha
- [ ] F1, sensibilidade e especificidade
- [ ] Melhor método por tipo de falha